In [1]:
import numpy as np
import pandas as pd 
import os
from pathlib import Path
import random
from sklearn.metrics.pairwise import cosine_similarity
import torch_sparse
from torch_geometric.utils import to_undirected
import torch 
from torch_geometric.utils import add_self_loops,add_remaining_self_loops
from torch_geometric.utils import remove_self_loops
from scipy import linalg
from torch_geometric.typing import SparseTensor
from typing import Callable, List, NamedTuple, Optional, Tuple, Union
from torch import Tensor
import torch_sparse
DEBUG = True 
import networkx as nx
from scipy import sparse, stats
from scipy.sparse import csgraph
from ipynb.fs.full.SpectralSparsifier import EffectiveResistance, LocalEffectiveResistance, get_sparse_adj_matrix
from scipy.sparse import csr_matrix
from scipy import linalg
from random import randint
import math 
from tqdm import tqdm
from time import time
import scipy.sparse as sp
import copy
from dgl.data import citation_graph as citegrh
from dgl import DGLGraph
from dgl import transforms
import networkx as nx
import os, sys
import pickle as pkl
import networkx as nx
import numpy as np
import scipy.sparse as sp
import pickle as pkl
from dgl import from_networkx, DGLGraph
device = torch.device("cpu")
from ipynb.fs.full.Dataset import get_data_from_dataset
from scipy import sparse
from scipy.sparse import csr_matrix
import numpy as np
import torch
import pandas as pd
from dgl import DGLGraph
import networkx as nx
import pandas as pd
from scipy import sparse, stats
from numpy import inf
import random
import dgl
from dgl import to_networkx, DGLGraph
from dgl import remove_self_loop
import scipy

Cpu count:  12


/home/muftiqur/Graph Research/Graph-Sparsification/GraphSparsification/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data directory:  ./Dataset/
Result directory: ./Dataset/RESULTS/

Dataset: Amherst41(1):
Number of graphs: 1
Number of features: 1193
Number of classes: 2

Data(x=[2235, 1193], edge_index=[2, 181908], y=[2235], train_mask=[2235], val_mask=[2235], test_mask=[2235])
Number of nodes: 2235
Number of edges: 181908
Average node degree: 81.39
Number of training nodes: 1341
Training node label rate: 0.60
Has isolated nodes: False
Has self-loops: False
Is undirected: True
Shifting label to non-negative


In [2]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
import random
from torch_geometric.datasets import WebKB
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from torch_geometric.datasets import Reddit, Reddit2
from torch_geometric.datasets import CoraFull

if os.uname()[1].find('gilbreth')==0: ##if not darwin(mac/locallaptop)
    DIR='/scratch/gilbreth/das90/Dataset/'
elif os.uname()[1].find('unimodular')==0:
    DIR='/scratch2/das90/Dataset/'
elif os.uname()[1].find('Siddharthas')==0:
    DIR='/Users/siddharthashankardas/Purdue/Dataset/'  
else:
    DIR='./Dataset/' 
Path(DIR).mkdir(parents=True, exist_ok=True)
RESULTS_DIR=DIR+'RESULTS/'
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
print("Data directory: ", DIR)
print("Result directory:", RESULTS_DIR)


Data directory:  ./Dataset/
Result directory: ./Dataset/RESULTS/


In [3]:
def get_data(DATASET_NAME):
        
    #DATASET_NAME='Cora' #"Cora", "CiteSeer", "PubMed"
    if DATASET_NAME in ["Cora", "CiteSeer", "PubMed"]:
        dataset = Planetoid(root=DIR+'Planetoid', name=DATASET_NAME, transform=NormalizeFeatures())

    elif DATASET_NAME == "Reddit2":
        dataset = Reddit2(root=DIR+'Reddit2', transform=NormalizeFeatures())

    elif DATASET_NAME == "Reddit":
        dataset = Reddit(root=DIR+'Reddit', transform=NormalizeFeatures())
    elif DATASET_NAME == "Texas":
        dataset = WebKB(root=DIR+'Texas', name="Texas",transform=NormalizeFeatures())
    elif DATASET_NAME == "Wisconsin":
            dataset = WebKB(root=DIR+'Wisconsin', name="Wisconsin",transform=NormalizeFeatures())
    elif DATASET_NAME == "Cornell":
         dataset = WebKB(root=DIR+'Cornell', name="Cornell",transform=NormalizeFeatures())
    else:    
        raise Exception('dataset not found')

    print()
    print(f'Dataset: {dataset}:')
    print('======================')
    print(f'Number of graphs: {len(dataset)}')
    print(f'Number of features: {dataset.num_features}')
    print(f'Number of classes: {dataset.num_classes}')

    data = dataset[0]  # Get the first graph object.

    print()
    print(data)
    print('===========================================================================================================')

    # Gather some statistics about the graph.
    print(f'Number of nodes: {data.num_nodes}')
    print(f'Number of edges: {data.num_edges}')
    print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
    print(f'Number of training nodes: {data.train_mask.sum()}')
    print(f'Training node label rate: {int(data.train_mask.sum()) / data.num_nodes:.2f}')
    print(f'Has isolated nodes: {data.has_isolated_nodes()}')
    print(f'Has self-loops: {data.has_self_loops()}')
    print(f'Is undirected: {data.is_undirected()}')
    
    return data, dataset

data,dataset = get_data_from_dataset('Cora')

Data directory:  ./Dataset/
Result directory: ./Dataset/RESULTS/
(Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708]), None)

Dataset: Cora():
Number of graphs: 1
Number of features: 1433
Number of classes: 7

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Number of nodes: 2708
Number of edges: 10556
Average node degree: 3.90
Number of training nodes: 140
Training node label rate: 0.05
Has isolated nodes: False
Has self-loops: False
Is undirected: True


In [4]:
def get_Adj_Matrix(edge_index: Union[Tensor, SparseTensor],
                 sizes: List[int], node_idx: Optional[Tensor] = None,
                 num_nodes: Optional[int] = None, return_e_id: bool = True,
                 transform: Callable = None,**kwargs):
    edge_index = edge_index.to('cpu')
    kwargs.pop('dataset', None)
    kwargs.pop('collate_fn', None)
    drop_last = kwargs.pop('drop_last', False)

    is_sparse_tensor = isinstance(edge_index, SparseTensor)
    __val__ = None
    
    if not is_sparse_tensor:
        if (num_nodes is None and node_idx is not None
                and node_idx.dtype == torch.bool):
            num_nodes = node_idx.size(0)
        if (num_nodes is None and node_idx is not None
                and node_idx.dtype == torch.long):
            num_nodes = max(int(edge_index.max()), int(node_idx.max())) + 1
        if num_nodes is None:
            num_nodes = int(edge_index.max()) + 1

        value = torch.arange(edge_index.size(1)) if return_e_id else None
        adj_t = SparseTensor(row=edge_index[0], col=edge_index[1],
                                    value=value,
                                    sparse_sizes=(num_nodes, num_nodes)).t()
    else:
        adj_t = edge_index
        if return_e_id:
            __val__ = adj_t.storage.value()
            value = torch.arange(adj_t.nnz())
            adj_t = adj_t.set_value(value, layout='coo')
        adj_t = adj_t

    adj_t.storage.rowptr()

    return adj_t

# Getting Cosine Weight List 
def precomputing_weight_calculation(x,edge_index: Union[Tensor, SparseTensor],
                 sizes: List[int], node_idx: Optional[Tensor] = None,
                 num_nodes: Optional[int] = None, return_e_id: bool = True,
                 transform: Callable = None,**kwargs):
        
        adj_t = get_Adj_Matrix(edge_index, node_idx = node_idx, sizes=sizes,num_nodes=num_nodes,return_e_id=return_e_id,transform=transform)
        #cosine_similarity_dictionary = Custom_CosineSimilarity_Sampling(x,edge_index)

        #weight = calculate_weights(adj_t, cosine_similarity_dictionary)
        return adj_t#torch.tensor(weight)

In [5]:
# #cosine_similarity_dictionary
def ER(u,v, L_inv,N):
    x_u = np.zeros((N,))
    x_v = np.zeros((N,)) 
    x_u[u] = 1
    x_v[v] = 1
    d_uv=x_u-x_v
    R_uv=d_uv.dot(L_inv.dot(d_uv)) ## (x_u-x_v)^T*L'*(x_u-x_v)
    return R_uv

def compute_ER(Adj, L_inv):
    start_nodes, end_nodes, weights = sparse.find(Adj)

    n = np.shape(Adj)[0]
    Reff = sparse.lil_matrix((n,n))
    for orig, end in zip(start_nodes, end_nodes):
        Reff[orig,end] = ER(orig, end, L_inv, n)  
    return Reff

def EffectiveResistance(Adj):
    #Adj = nx.adjacency_matrix(G)
    L, D  = csgraph.laplacian(Adj, normed=False, return_diag=True)
    #print(np.allclose(L.todense(), np.diag(D)-Adj)) #verify L=D-A 
    L_inv = linalg.pinv(L.todense())
    Reff=compute_ER(Adj, L_inv)
    return Reff

def ERprob(Adjacency_matrix,compute='exact'):
    start_nodes, end_nodes, weights = sparse.find(Adjacency_matrix)
    N=Adjacency_matrix.shape[0]
    if DEBUG: print("Computing edge resistances: ... ",compute)
    if compute=='exact':
        Re=EffectiveResistance(Adjacency_matrix)
        Re = np.maximum(0, Re[start_nodes, end_nodes].toarray())
        print("FINISHED COMPUTING")
        ER_weight_matrix = np.zeros((N,N))
        for i in range(len(start_nodes)):
            ER_weight_matrix[end_nodes[i],start_nodes[i]] = Re[0][i]
            ER_weight_matrix[start_nodes[i],end_nodes[i]] = Re[0][i]
    else:
        print("Computing Local Effective Resistance")
        Re=LocalEffectiveResistance(method=None, eps=0.4, lmbda=0.1)
    if DEBUG: print("Finished computing resistances:")

    return ER_weight_matrix

def create_adjacency_matrix(edge_index, num_nodes, scipy_fmt='csr'):
    #adj_matrix = torch.zeros((num_nodes, num_nodes), dtype=torch.float)
    edge_index = remove_self_loops(edge_index)[0]
    adj_mat = torch.zeros((num_nodes,num_nodes))
    edges = edge_index.t()
    adj_mat[edges[:,0], edges[:,1]] = 1
    adj_mat[edges[:,1], edges[:,0]] = 1
    #adj_mat[17,14] = 1
    print(adj_mat)
    if scipy_fmt == 'csr':
        adj_matrix_csr = csr_matrix(adj_mat)
        return adj_matrix_csr
    else:
        raise ValueError(f"Unsupported format: {scipy_fmt}")

def compute_effective_resistance_(edge_index,num_nodes,scipy_fmt='csr', effective_resistance_algorithm='exact'):
    edge_index = remove_self_loops(edge_index)[0]
    adj_mat = torch.zeros((num_nodes,num_nodes))
    edges = edge_index.t()
    print("Computing Adj matrix")
    adj_mat[edges[:,0], edges[:,1]] = 1
    adj_mat[edges[:,1], edges[:,0]] = 1
    print("Finished computing Adj Matrix")
    
    if scipy_fmt == 'csr':
        sparse_adj = csr_matrix(adj_mat)
    else:
        raise ValueError(f"Unsupported format: {scipy_fmt}")
    ER_weight_matrix = ERprob(sparse_adj, compute=effective_resistance_algorithm)
    return ER_weight_matrix

In [6]:
def get_ER_matrix(x, edge_index,sizes):
    adj_t = precomputing_weight_calculation(x,edge_index,sizes)
    rowptr, col , _ = adj_t.csr()
    Adj = np.zeros((len(rowptr) - 1, len(rowptr) - 1))
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            Adj[source_vertex, target_vertex] =  1
    sparse_adj = csr_matrix(Adj)
    WSP = ERprob(sparse_adj)

    rowptr,col,_ = adj_t.csr()
    weights = []
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            edge_weight = WSP[i][target_vertex]
            weights.append(edge_weight)
    print("Finish Computing True Effective Resistance")
    
    return weights

In [7]:
from torch_geometric.utils import remove_self_loops

def generate_L(Ag, N):
    v = np.ones(N)
    Dv = Ag.dot(v)
    Dg = csr_matrix((Dv, (np.arange(N), np.arange(N))), shape=(N, N))
    Lg = Dg - Ag
    print("Finished building the Laplacian")
    return Lg

def compute_reff(W, V):
    start_nodes, end_nodes, weights = sparse.find(sparse.tril(W))
    n = np.shape(W)[0]
    Reff = sparse.lil_matrix((n, n))
    for orig, end in zip(start_nodes, end_nodes):
        Reff[orig, end] = np.linalg.norm(V[orig, :] - V[end, :]) ** 2
    return Reff

def graph_sparsify(Lg, epsilon,filename, choice=1):
    print("Computing resistances for ", filename)
    N = np.size(Lg, 0)
    Dv = Lg.diagonal()
    print(N)
    Dg = csr_matrix((Dv, (np.arange(N), np.arange(N))), shape=(N, N))
    W = Dg - Lg
    read_V = [1, 3, 4, 5, 8, 9, 10, 11, 12, 13]
    if choice in read_V:
        print("Reading V matrix:..")
        V_frame = pd.read_csv(filename, header=None)
        V = V_frame.to_numpy()
        print("Computing edge resistances:... ")

        resistance_distances = compute_reff(W, V)
        print("Finished loading resistances:")
    else:
        resistance_distances = np.loadtxt(filename)
    start_nodes, end_nodes, weights = sparse.find(sparse.tril(W))
    weights = np.maximum(0, weights)
    if choice in read_V:
        Re = np.maximum(0, resistance_distances[start_nodes, end_nodes].toarray())
        ER_weight_matrix = np.zeros((N,N))
        for i in range(len(start_nodes)):
            ER_weight_matrix[end_nodes[i],start_nodes[i]] = Re[0][i]
            ER_weight_matrix[start_nodes[i],end_nodes[i]] = Re[0][i]
    else:
        Re = np.maximum(0, resistance_distances[start_nodes, end_nodes])

    # Pe = weights * Re
    # Pe = Pe / np.sum(Pe)
    # Pe = np.squeeze(Pe)
    # C0 = 1 / 30.
    # C = 4 * C0
    # q = round(N * np.log(N) * 9 * C ** 2 / (epsilon ** 2))
    # random.seed(42)
    # results = np.random.choice(np.arange(np.shape(Pe)[0]), int(q), p=list(Pe))
    # spin_counts = np.unique(results, return_counts=True)
    # per_spin_weights = weights / (q * Pe)
    # per_spin_weights[per_spin_weights == inf] = 0
    # counts = np.zeros(np.shape(weights)[0]) 
    # counts[spin_counts[0]] = spin_counts[1]
    # new_weights = counts * per_spin_weights
    # sparserW = sparse.csc_matrix((np.squeeze(new_weights), (start_nodes, end_nodes)),
    #                              shape=(N, N))
    # sparserW = sparserW + sparserW.T
    return ER_weight_matrix

def save_weight_list(data, dataset_name, Wsp):
    edge_index = remove_self_loops(data.edge_index)[0]
    weight_list = np.zeros(len(edge_index[0]))
    w_matrix = Wsp.toarray()
    for ind,(u,v) in enumerate(zip(data.edge_index[0],data.edge_index[1])):
        weight_list[ind] = w_matrix[u][v]
    weight_tensor = torch.tensor(weight_list)
    torch.save(weight_tensor, dataset_name + '_weight_list.pt')

def adjacency_matrix(data, num_nodes, scipy_fmt='csr'):
    adj_matrix = torch.zeros((num_nodes, num_nodes), dtype=torch.float)
    edge_index = remove_self_loops(data.edge_index)[0]
    for i in range(edge_index.shape[1]):
        source = edge_index[0, i].item()
        target = edge_index[1, i].item()
        adj_matrix[source, target] = 1
        adj_matrix[target, source] = 1
    
    # Convert the PyTorch tensor to a NumPy array
    adj_matrix_np = adj_matrix.numpy()
    
    # Convert the NumPy array to a SciPy CSR matrix if requested
    if scipy_fmt == 'csr':
        adj_matrix_csr = csr_matrix(adj_matrix_np)
        return adj_matrix_csr
    else:
        raise ValueError(f"Unsupported format: {scipy_fmt}")

def generate_spare_graph_(Ag, N, filename, epsilon=0.6):
    #Ag = g.adj_external(scipy_fmt = 'csr')
    Lg = generate_L(Ag, N)
    print("Sparsifying the graph: ")
    Wsp = graph_sparsify(Lg, epsilon, filename)
    #Re = graph_sparsify(Lg, epsilon, filename)
    print("Finished sparsifying the graph: ")
    #adj_matrix = g_sp.adjacency_matrix().to_dense()
    # add self loop
    #g.add_edges(g.nodes(), g.nodes())
    #g_sp.add_edges(g_sp.nodes(), g_sp.nodes())
    return Wsp

In [11]:
def get_approximate_ER_matrix(x,edge_index,N,sizes,filename):
    adj_t = precomputing_weight_calculation(x,edge_index,sizes)
    rowptr, col , _ = adj_t.csr()
    Adj = np.zeros((len(rowptr) - 1, len(rowptr) - 1))
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            Adj[source_vertex, target_vertex] =  1
    sparse_adj = csr_matrix(Adj)
    WSP = generate_spare_graph_(sparse_adj, N, filename)
    rowptr,col,_ = adj_t.csr()
    weights = []
    for i in range(len(rowptr) - 1):
        rowstart = rowptr[i]
        rowend = rowptr[i + 1]
        edge_list = col[rowstart:rowend]
        for target_vertex in edge_list:
            source_vertex = i
            edge_weight = WSP[i][target_vertex]
            weights.append(edge_weight)
    print("Finish Computing True Effective Resistance")
    return weights

In [14]:
# N = data.num_nodes
# filename = 'V_Cora.csv'
# approx_er_matrix = get_approximate_ER_matrix(data.x, data.edge_index,N=N,sizes=[8,4],filename=filename)